# 処理時間を計測する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/A_colab_python_tips/timing.ipynb)

NLP クックブック A-07 ／ Colab と Python の小技

どの処理が遅いのかを知りたい。2 つの実装のどちらが速いかを比べたい。

In [1]:
%%time
# セル全体の実行時間を測る (%%time はセルの 1 行目に書く)
total = sum(i * i for i in range(3_000_000))

CPU times: user 136 ms, sys: 0 ns, total: 136 ms
Wall time: 183 ms


In [2]:
import re

text = "自然言語処理 " * 10000
pattern = re.compile(r"\S+")

# %timeit: 何度も繰り返して平均と標準偏差を出す。短い処理の比較に向く
%timeit text.split()
%timeit pattern.findall(text)

316 μs ± 23.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
1.29 ms ± 29.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [3]:
import time
from contextlib import contextmanager

@contextmanager
def timer(name):
    """with ブロックの時間を測る。スクリプトに書き写しても使える"""
    start = time.perf_counter()
    yield
    print(f"[{name}] {time.perf_counter() - start:.3f} 秒")

with timer("トークン化"):
    tokens = text.split()
with timer("頻度の集計"):
    from collections import Counter
    freq = Counter(tokens)

[トークン化] 0.000 秒
[頻度の集計] 0.000 秒


In [4]:
# どの関数に時間がかかっているかを調べる (プロファイラ)
import cProfile, pstats

def slow_pipeline():
    words = [w for w in text.split()]
    return sorted(set(words)), sum(len(w) for w in words), re.findall(r"処理", text)

profiler = cProfile.Profile(); profiler.runcall(slow_pipeline)
_ = pstats.Stats(profiler).sort_stats("cumulative").print_stats(5)

         20099 function calls (20098 primitive calls) in 0.007 seconds

   Ordered by: cumulative time
   List reduced from 46 to 5 due to restriction <5>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.001    0.001    0.007    0.007 /tmp/ipykernel_3005/1592709111.py:4(slow_pipeline)
        1    0.001    0.001    0.005    0.005 {built-in method builtins.sum}
    10001    0.002    0.000    0.003    0.000 /tmp/ipykernel_3005/1592709111.py:6(<genexpr>)
        1    0.000    0.000    0.001    0.001 /usr/local/lib/python3.13/re/__init__.py:270(findall)
10014/10013    0.001    0.000    0.001    0.000 {built-in method builtins.len}




- GPU の処理は非同期なので、正確に測るには計測の前後で `torch.cuda.synchronize()` を呼ぶ。
- 初回の実行は、モデルの読み込みやキャッシュの作成で遅い。2 回目以降を測る。
- Colab の無料枠は実行のたびに CPU や GPU の割り当てが変わる。異なるセッションで測った時間は比較できない。